# FloodCastNet — Phase 1: Architecture
**Version:** 1.0.0
**Date:** March 2026
**Author:** Abu Sameer

## What This Notebook Contains
Complete from-scratch implementation of FloodCastNet architecture.

| Cell | Content | Status |
|------|---------|--------|
| 1 | Imports + Environment + Config | ✅ |
| 2 | All Model Modules (ConvLSTM, ViT, TFT, GNN, Fusion) | ✅ |
| 3 | End-to-End Memory Test | ✅ |
| 4 | SP-1: Flood Map Decoder | ✅ |
| 5 | SP-2: Severity Forecast | ✅ |
| 6 | SP-3 + SP-4: Risk + Damage | ✅ |
| 7 | SP-5: Evacuation GNN | ✅ |
| 8 | SP-6: RL Resource Allocation | ✅ |
| 9 | SP-7 + SP-8 + SP-9 | ✅ |
| 10 | FloodCastNet Complete Assembly | ✅ |
| 11 | Physics Loss + MultiTask Loss | ✅ |
| 12 | Training Loop + DataLoader | ✅ |
| 13 | First Training Run (3 epochs) | ✅ |
| 14 | Loss Fix + Model Save | ✅ |

## Key Results
- Total params: 4,660,313
- GPU memory: 1.66 GB (T4 x2)
- Loss: 2.9215 → 2.8791 (going down)
- Model size: 18.8 MB
- Outputs: 14 (all 9 sub-problems)

## Architecture Summary
- SpatioTemporal Encoder: ConvLSTM(32,64,128) + ViT
- Temporal Encoder: TFT-based (hidden=64)
- Static Encoder: ResNet blocks (base=32)
- Fusion: CrossModal Transformer (dim=128, pool=8x8)
- 9 Sub-Problem Heads: SP1-SP9
- Loss: Physics-informed multi-task

In [1]:
import subprocess, sys, torch, random, os, json
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from einops import rearrange
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}  : {p.name}  {p.total_memory/1e9:.1f}GB")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU 0  : Tesla T4  15.6GB
GPU 1  : Tesla T4  15.6GB


In [2]:
@dataclass
class DataConfig:
    satellite_bands   : List[str] = field(default_factory=lambda: ["VV","VH","VV_VH_ratio","NDWI"])
    image_size        : Tuple[int,int] = (256, 256)
    sequence_length   : int = 12
    weather_features  : List[str] = field(default_factory=lambda: [
        "total_precipitation","2m_temperature","surface_pressure",
        "10m_u_wind","10m_v_wind","soil_water_layer_1","runoff"])
    weather_lookback  : int = 72
    forecast_horizons : List[int] = field(default_factory=lambda: [6,24,48,72])
    gauge_features    : List[str] = field(default_factory=lambda: ["stream_flow","gauge_height","velocity"])
    train_ratio: float = 0.7
    val_ratio  : float = 0.15
    test_ratio : float = 0.15
    normalize  : bool  = True

@dataclass
class ModelConfig:
    sat_in_channels      : int       = 4
    convlstm_hidden_dims : List[int] = field(default_factory=lambda: [32, 64, 128])
    convlstm_kernel_size : int       = 3
    vit_patch_size       : int       = 8
    vit_heads            : int       = 4
    vit_depth            : int       = 2
    vit_mlp_dim          : int       = 256
    tft_hidden_dim       : int       = 64
    tft_lstm_layers      : int       = 2
    tft_attention_heads  : int       = 4
    tft_dropout          : float     = 0.1
    gnn_hidden_dim       : int       = 64
    gnn_layers           : int       = 3
    gnn_heads            : int       = 4
    static_hidden_dim    : int       = 32
    static_resnet_blocks : int       = 4
    fusion_dim           : int       = 128
    fusion_heads         : int       = 4
    fusion_dropout       : float     = 0.1
    fusion_depth         : int       = 2
    fusion_pool_size     : int       = 8
    flood_map_channels   : int       = 1
    risk_classes         : int       = 4
    damage_classes       : int       = 3
    route_features       : int       = 32
    rl_state_dim         : int       = 256
    rl_action_dim        : int       = 64
    mc_dropout_samples   : int       = 50
    climate_years        : List[int] = field(default_factory=lambda: [2030,2040,2050])

@dataclass
class TrainingConfig:
    batch_size                  : int   = 2
    gradient_accumulation_steps : int   = 16   # effective batch = 32
    epochs                      : int   = 100
    optimizer                   : str   = "AdamW"
    learning_rate               : float = 3e-4
    weight_decay                : float = 1e-4
    scheduler                   : str   = "CosineAnnealingWarmRestarts"
    warmup_epochs               : int   = 5
    use_amp                     : bool  = True
    gradient_checkpointing      : bool  = True
    loss_weights: Dict[str,float] = field(default_factory=lambda: {
        "sp1_flood_map"  : 1.0, "sp2_severity"   : 0.8,
        "sp3_risk"       : 0.6, "sp4_damage"      : 0.5,
        "sp5_routes"     : 0.4, "sp6_rl"          : 0.3,
        "sp7_causal"     : 0.3, "sp8_uncertainty" : 0.2,
        "sp9_climate"    : 0.3, "physics"         : 0.1,
    })
    save_every_n_epochs    : int = 5
    keep_top_k_checkpoints : int = 3

@dataclass
class MasterConfig:
    data     : DataConfig     = field(default_factory=DataConfig)
    model    : ModelConfig    = field(default_factory=ModelConfig)
    training : TrainingConfig = field(default_factory=TrainingConfig)
    project_name    : str = "FloodCastNet"
    version         : str = "1.0.0"
    seed            : int = 42
    root_dir        : str = "/kaggle/working/FloodCastNet"
    checkpoint_dir  : str = "/kaggle/working/FloodCastNet/checkpoints"
    log_dir         : str = "/kaggle/working/FloodCastNet/logs"

config = MasterConfig()

def set_seed(seed):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(config.seed)

print(f"fusion_dim   : {config.model.fusion_dim}  (128, down from 256)")
print(f"convlstm dims: {config.model.convlstm_hidden_dims}  (32/64/128, down from 64/128/256)")
print(f"AMP          : {config.training.use_amp}")
print(f"eff batch    : {config.training.batch_size * config.training.gradient_accumulation_steps}")

fusion_dim   : 128  (128, down from 256)
convlstm dims: [32, 64, 128]  (32/64/128, down from 64/128/256)
AMP          : True
eff batch    : 32


In [3]:
# ── ConvLSTM ─────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, ks):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.gates     = nn.Conv2d(in_ch+hidden_ch, 4*hidden_ch, ks, padding=ks//2, bias=True)
        self.cell_norm = nn.LayerNorm([hidden_ch])

    def forward(self, x, h, c):
        i,f,o,g = self.gates(torch.cat([x,h], dim=1)).chunk(4, dim=1)
        i,f,o   = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        c_next  = f*c + i*torch.tanh(g)
        B,C,H,W = c_next.shape
        c_n     = self.cell_norm(c_next.permute(0,2,3,1)).permute(0,3,1,2)
        return o*torch.tanh(c_n), c_next

    def init_hidden(self, B, H, W, device):
        z = lambda: torch.zeros(B, self.hidden_ch, H, W, device=device)
        return z(), z()


class ConvLSTM(nn.Module):
    def __init__(self, in_ch, hidden_dims, ks):
        super().__init__()
        dims = [in_ch] + hidden_dims
        self.cells      = nn.ModuleList([
            ConvLSTMCell(dims[i], dims[i+1], ks) for i in range(len(hidden_dims))
        ])
        self.hidden_dims = hidden_dims

    def forward(self, x):
        B,T,C,H,W = x.shape
        states = [c.init_hidden(B,H,W,x.device) for c in self.cells]
        for t in range(T):
            inp = x[:,t]
            for i,cell in enumerate(self.cells):
                h,c       = states[i]
                h,c       = cell(inp, h, c)
                states[i] = (h, c)
                inp       = h
        # return only final hidden state -- saves T-1 activation buffers
        return h    # (B, hidden_dims[-1], H, W)


# ── ViT (applied ONCE on final ConvLSTM state) ───────────────
class ViTBottleneck(nn.Module):
    def __init__(self, ch, patch_size, heads, depth, mlp_dim):
        super().__init__()
        self.ps        = patch_size
        self.patch_proj = nn.Conv2d(ch, ch, patch_size, stride=patch_size)
        self.pos_emb   = nn.Parameter(torch.randn(1, 256, ch) * 0.02)
        enc_layer      = nn.TransformerEncoderLayer(
            d_model=ch, nhead=heads, dim_feedforward=mlp_dim,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, depth)
        self.unproj    = nn.ConvTranspose2d(ch, ch, patch_size, stride=patch_size)
        self.norm      = nn.LayerNorm(ch)

    def forward(self, x):
        B,C,H,W = x.shape
        p       = self.patch_proj(x)
        ph,pw   = p.shape[2], p.shape[3]
        seq     = rearrange(p, 'b c h w -> b (h w) c')
        seq     = self.norm(seq + self.pos_emb[:, :seq.shape[1]])
        seq     = self.transformer(seq)
        p_out   = rearrange(seq, 'b (h w) c -> b c h w', h=ph, w=pw)
        return self.unproj(p_out)


# ── SpatioTemporal Encoder ────────────────────────────────────
class SpatioTemporalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.convlstm = ConvLSTM(
            cfg.model.sat_in_channels,
            cfg.model.convlstm_hidden_dims,
            cfg.model.convlstm_kernel_size
        )
        D = cfg.model.convlstm_hidden_dims[-1]   # 128
        self.vit = ViTBottleneck(
            D, patch_size=4, heads=cfg.model.vit_heads,
            depth=cfg.model.vit_depth, mlp_dim=cfg.model.vit_mlp_dim
        )
        self.res_proj = nn.Conv2d(D, D, 1)
        self.out_dim  = D

    def forward(self, x):
        # x: (B,T,C,H,W)
        h    = self.convlstm(x)          # (B, 128, H, W) -- final state only
        vout = self.vit(h)
        return vout + self.res_proj(h)   # (B, 128, H, W)


# ── GRN + VSN + Temporal Encoder ─────────────────────────────
class GatedResidualNetwork(nn.Module):
    def __init__(self, in_d, hid_d, out_d, drop=0.1):
        super().__init__()
        self.fc1  = nn.Linear(in_d, hid_d)
        self.fc2  = nn.Linear(hid_d, out_d)
        self.gate = nn.Linear(hid_d, out_d)
        self.skip = nn.Linear(in_d, out_d) if in_d != out_d else nn.Identity()
        self.norm = nn.LayerNorm(out_d)
        self.drop = nn.Dropout(drop)
        self.elu  = nn.ELU()
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='linear')
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='linear')
        nn.init.constant_(self.gate.bias, -2.0)
        with torch.no_grad(): self.fc2.weight.mul_(0.1)

    def forward(self, x):
        res = self.skip(x)
        h   = self.drop(self.elu(self.fc1(x)))
        g   = torch.sigmoid(self.gate(h))
        return self.norm(g * self.fc2(h) + (1-g) * res)


class VariableSelectionNetwork(nn.Module):
    def __init__(self, n_vars, var_dim, hid, drop=0.1):
        super().__init__()
        self.var_grns  = nn.ModuleList([
            GatedResidualNetwork(var_dim, hid, hid, drop) for _ in range(n_vars)
        ])
        self.select_grn = GatedResidualNetwork(n_vars*var_dim, hid, n_vars, drop)
        self.highway   = nn.Linear(n_vars*var_dim, hid)
        self.hw_gate   = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        B,T,V,D = x.shape
        outs = [self.var_grns[i](x[:,:,i,:].reshape(B*T,D)).reshape(B,T,-1)
                for i in range(V)]
        stack   = torch.stack(outs, dim=2)
        x_flat  = x.reshape(B*T, V*D)
        weights = torch.softmax(
            self.select_grn(x_flat).reshape(B,T,V,1), dim=2
        )
        weighted = (stack * weights).sum(dim=2)
        hw  = self.highway(x_flat).reshape(B,T,-1)
        mix = torch.sigmoid(self.hw_gate)
        return mix*weighted + (1-mix)*hw, weights.squeeze(-1)


class TemporalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        n_vars = len(cfg.data.weather_features) + len(cfg.data.gauge_features)
        H      = cfg.model.tft_hidden_dim    # 64
        self.input_proj = nn.Linear(1, H)
        self.var_select = VariableSelectionNetwork(n_vars, H, H, cfg.model.tft_dropout)
        self.lstm       = nn.LSTM(H, H, cfg.model.tft_lstm_layers,
                                  batch_first=True,
                                  dropout=cfg.model.tft_dropout if cfg.model.tft_lstm_layers>1 else 0)
        self.attn       = nn.MultiheadAttention(H, cfg.model.tft_attention_heads,
                                                dropout=cfg.model.tft_dropout, batch_first=True)
        self.gate_skip  = GatedResidualNetwork(H, H, H, cfg.model.tft_dropout)
        self.norm1      = nn.LayerNorm(H)
        self.norm2      = nn.LayerNorm(H)
        self.out_dim    = H

    def forward(self, weather, gauge):
        B,T,_ = weather.shape
        x     = torch.cat([weather, gauge], dim=-1).unsqueeze(-1)  # (B,T,10,1)
        x     = self.input_proj(x)                                  # (B,T,10,H)
        sel, weights = self.var_select(x)
        lstm_out, _  = self.lstm(sel)
        lstm_out     = self.norm1(lstm_out)
        attn_out, _  = self.attn(lstm_out, lstm_out, lstm_out)
        out          = self.norm2(self.gate_skip(lstm_out + attn_out))
        return out, weights


# ── Static Map Encoder ────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, ch, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch)
        )
        self.relu = nn.ReLU(True)

    def forward(self, x): return self.relu(self.net(x) + x)


class StaticMapEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        base = cfg.model.static_hidden_dim   # 32
        self.stem   = nn.Sequential(
            nn.Conv2d(5, base, 7, padding=3, bias=False),
            nn.BatchNorm2d(base), nn.ReLU(True)
        )
        self.stage1 = nn.Sequential(*[ResidualBlock(base) for _ in range(2)])
        self.down1  = nn.Sequential(
            nn.Conv2d(base, base*2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base*2), nn.ReLU(True)
        )
        self.stage2 = nn.Sequential(*[ResidualBlock(base*2) for _ in range(2)])
        self.proj   = nn.Conv2d(base*2, 128, 1)  # project to 128 to match sat encoder
        self.s_attn = nn.Sequential(
            nn.Conv2d(128, 32, 1), nn.ReLU(True), nn.Conv2d(32, 1, 1), nn.Sigmoid()
        )
        self.out_dim = 128

    def forward(self, x):
        x    = self.stage1(self.stem(x))
        x    = self.stage2(self.down1(x))
        x    = self.proj(x)
        x    = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        attn = self.s_attn(x)
        return x * attn, attn


# ── Shape Aligner ─────────────────────────────────────────────
class ShapeAligner(nn.Module):
    def __init__(self, fusion_dim, pool_size=8):
        super().__init__()
        D, P = fusion_dim, pool_size
        self.pool_size  = P
        self.sat_proj   = nn.Sequential(nn.Linear(128, D), nn.LayerNorm(D))
        self.temp_proj  = nn.Sequential(nn.Linear(64,  D), nn.LayerNorm(D), nn.GELU())
        self.stat_proj  = nn.Sequential(nn.Linear(128, D), nn.LayerNorm(D))
        self.sat_pos    = nn.Parameter(torch.randn(1, P*P, D) * 0.02)
        self.stat_pos   = nn.Parameter(torch.randn(1, P*P, D) * 0.02)
        self.sat_type   = nn.Parameter(torch.randn(1, 1, D) * 0.02)
        self.temp_type  = nn.Parameter(torch.randn(1, 1, D) * 0.02)
        self.stat_type  = nn.Parameter(torch.randn(1, 1, D) * 0.02)

    def forward(self, sat_feat, temp_feat, stat_feat):
        P   = self.pool_size
        sat = F.adaptive_avg_pool2d(sat_feat, (P,P)).flatten(2).transpose(1,2)
        sat = self.sat_proj(sat) + self.sat_pos + self.sat_type
        tmp = temp_feat.mean(dim=1)
        tmp = self.temp_proj(tmp).unsqueeze(1) + self.temp_type
        st  = F.adaptive_avg_pool2d(stat_feat, (P,P)).flatten(2).transpose(1,2)
        st  = self.stat_proj(st) + self.stat_pos + self.stat_type
        return sat, tmp, st, (P,P)


# ── Cross-Modal Fusion ────────────────────────────────────────
class CrossModalFusionLayer(nn.Module):
    def __init__(self, D, heads, drop):
        super().__init__()
        mha = lambda: nn.MultiheadAttention(D, heads, dropout=drop, batch_first=True)
        self.sat_x  = mha(); self.temp_x = mha(); self.stat_x = mha()
        self.sg  = nn.Parameter(torch.zeros(1))
        self.tg  = nn.Parameter(torch.zeros(1))
        self.stg = nn.Parameter(torch.zeros(1))
        ffn = lambda: nn.Sequential(
            nn.Linear(D, D*4), nn.GELU(), nn.Dropout(drop),
            nn.Linear(D*4, D), nn.Dropout(drop)
        )
        self.sat_ffn  = ffn(); self.temp_ffn = ffn(); self.stat_ffn = ffn()
        for tag in ['s1','s2','t1','t2','st1','st2']:
            setattr(self, f'n{tag}', nn.LayerNorm(D))

    def forward(self, sat, temp, stat):
        ctx = lambda a,b: torch.cat([a,b], dim=1)
        sx,_ = self.sat_x(sat,  ctx(temp,stat), ctx(temp,stat))
        sat  = self.ns1(sat  + torch.sigmoid(self.sg)  * sx)
        sat  = self.ns2(sat  + self.sat_ffn(sat))
        tx,_ = self.temp_x(temp, ctx(sat,stat),  ctx(sat,stat))
        temp = self.nt1(temp + torch.sigmoid(self.tg)  * tx)
        temp = self.nt2(temp + self.temp_ffn(temp))
        stx,_= self.stat_x(stat, ctx(sat,temp),  ctx(sat,temp))
        stat = self.nst1(stat + torch.sigmoid(self.stg) * stx)
        stat = self.nst2(stat + self.stat_ffn(stat))
        return sat, temp, stat


class CrossModalFusion(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg.model.fusion_dim
        self.aligner = ShapeAligner(D, pool_size=cfg.model.fusion_pool_size)
        self.layers  = nn.ModuleList([
            CrossModalFusionLayer(D, cfg.model.fusion_heads, cfg.model.fusion_dropout)
            for _ in range(cfg.model.fusion_depth)
        ])
        self.norm = nn.LayerNorm(D)

    def forward(self, sat, temp, stat):
        sat, temp, stat, hw = self.aligner(sat, temp, stat)
        for l in self.layers:
            sat, temp, stat = l(sat, temp, stat)
        return self.norm(sat), self.norm(temp), self.norm(stat), hw


print("All modules defined cleanly")

All modules defined cleanly


In [4]:
torch.cuda.empty_cache()

encoder    = SpatioTemporalEncoder(config).cuda()
temp_enc   = TemporalEncoder(config).cuda()
static_enc = StaticMapEncoder(config).cuda()
fusion     = CrossModalFusion(config).cuda()

scaler = torch.cuda.amp.GradScaler()   # AMP scaler

B = 2
sat_in  = torch.randn(B, 12, 4, 64, 64).cuda()
w_in    = torch.randn(B, 72,  7).cuda()
g_in    = torch.randn(B, 72,  3).cuda()
st_in   = torch.randn(B,  5, 64, 64).cuda()

print("Memory at each stage (AMP fp16):")
torch.cuda.empty_cache()
m0 = torch.cuda.memory_allocated(0)/1e9
print(f"  models loaded     : {m0:.2f} GB")

# AMP forward
with torch.cuda.amp.autocast():
    sat_feat           = encoder(sat_in)
    temp_feat, _       = temp_enc(w_in, g_in)
    stat_feat, _       = static_enc(st_in)
    so, to, sto, (H,W) = fusion(sat_feat, temp_feat, stat_feat)
    loss = so.mean() + to.mean() + sto.mean()

m1 = torch.cuda.memory_allocated(0)/1e9
print(f"  after forward     : {m1:.2f} GB")

scaler.scale(loss).backward()

m2 = torch.cuda.memory_allocated(0)/1e9
print(f"  after backward    : {m2:.2f} GB")
print(f"  remaining         : {15.6-m2:.2f} GB")
print()
print(f"Output shapes into decoders:")
print(f"  sat  : {so.shape}")
print(f"  temp : {to.shape}")
print(f"  stat : {sto.shape}")

total = sum(sum(p.numel() for p in m.parameters())
            for m in [encoder,temp_enc,static_enc,fusion])
print(f"\nTotal params: {total:,}")

if 15.6 - m2 >= 2.5:
    print("Memory OK -- decoders build kar sakte hain")
else:
    print("Still tight -- batao aur optimize karte hain")

/tmp/ipykernel_24/1436218676.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, depth)
/tmp/ipykernel_24/2127905747.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()   # AMP scaler
/tmp/ipykernel_24/2127905747.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Memory at each stage (AMP fp16):
  models loaded     : 0.02 GB
  after forward     : 0.52 GB
  after backward    : 0.06 GB
  remaining         : 15.54 GB

Output shapes into decoders:
  sat  : torch.Size([2, 64, 128])
  temp : torch.Size([2, 1, 128])
  stat : torch.Size([2, 64, 128])

Total params: 3,772,634
Memory OK -- decoders build kar sakte hain


In [5]:
# SP-1: pixel-level flood probability map
# input : sat tokens (B, 64, 128) = 8x8 spatial grid
# output: (B, 1, 64, 64) flood probability per pixel
#
# challenge: 8x8 -> 64x64 = 8x upsampling
# U-Net style: upsample in stages, double resolution each time
# 8x8 -> 16x16 -> 32x32 -> 64x64

class FloodMapDecoder(nn.Module):

    def __init__(self, fusion_dim, pool_size=8):
        super().__init__()
        D = fusion_dim   # 128
        P = pool_size    # 8

        self.reshape_proj = nn.Linear(D, D)

        # upsample block: doubles H and W each time
        def up_block(in_ch, out_ch):
            return nn.Sequential(
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(True)
            )

        # 8x8 -> 16x16 -> 32x32 -> 64x64
        self.up1 = up_block(D, 64)      # 8  -> 16
        self.up2 = up_block(64, 32)     # 16 -> 32
        self.up3 = up_block(32, 16)     # 32 -> 64

        # final head: logit per pixel (no sigmoid -- BCEWithLogitsLoss handles it)
        self.head = nn.Conv2d(16, 1, kernel_size=1)

        # uncertainty branch: predicts aleatoric uncertainty per pixel
        # separate from MC dropout epistemic uncertainty (SP-8)
        self.uncert_head = nn.Sequential(
            nn.Conv2d(16, 8, 3, padding=1),
            nn.ReLU(True),
            nn.Conv2d(8, 1, 1),
            nn.Softplus()    # always positive -- it's a variance estimate
        )

        self.pool_size = P

    def forward(self, sat_tokens):
        # sat_tokens: (B, P*P, D) = (B, 64, 128)
        B, N, D = sat_tokens.shape
        P = self.pool_size

        x = self.reshape_proj(sat_tokens)          # (B, 64, 128)
        x = x.transpose(1, 2).reshape(B, D, P, P) # (B, 128, 8, 8)

        x  = self.up1(x)    # (B, 64,  16, 16)
        x  = self.up2(x)    # (B, 32,  32, 32)
        x  = self.up3(x)    # (B, 16,  64, 64)

        flood_logits = self.head(x)        # (B, 1, 64, 64)
        uncertainty  = self.uncert_head(x) # (B, 1, 64, 64)

        return flood_logits, uncertainty


# ── test ──────────────────────────────────────────────────────
sp1 = FloodMapDecoder(
    fusion_dim=config.model.fusion_dim,
    pool_size=config.model.fusion_pool_size
).cuda()

sat_tokens = torch.randn(2, 64, 128).cuda()

with torch.no_grad():
    logits, uncert = sp1(sat_tokens)

print("SP-1 FloodMapDecoder:")
print(f"  input   : {sat_tokens.shape}")
print(f"  logits  : {logits.shape}   <-- (2, 1, 64, 64)")
print(f"  uncert  : {uncert.shape}   <-- aleatoric uncertainty")
print(f"  logit range : [{logits.min():.2f}, {logits.max():.2f}]")
print(f"  uncert range: [{uncert.min():.2f}, {uncert.max():.2f}]  <-- must be > 0")
print(f"  params  : {sum(p.numel() for p in sp1.parameters()):,}")
assert logits.shape == (2, 1, 64, 64)
assert uncert.min() > 0, "uncertainty must be positive"
print("  PASSED")

SP-1 FloodMapDecoder:
  input   : torch.Size([2, 64, 128])
  logits  : torch.Size([2, 1, 64, 64])   <-- (2, 1, 64, 64)
  uncert  : torch.Size([2, 1, 64, 64])   <-- aleatoric uncertainty
  logit range : [-1.27, 2.95]
  uncert range: [0.42, 1.11]  <-- must be > 0
  params  : 109,650
  PASSED


In [6]:
# SP-2: how severe will flood be at 6h / 24h / 48h / 72h
# input : temp token (B, 1, 128) + sat tokens (B, 64, 128)
# output: severity score per horizon (B, 4) -- one per forecast horizon

class SeverityForecastHead(nn.Module):

    def __init__(self, fusion_dim, horizons):
        super().__init__()
        D = fusion_dim       # 128
        H = len(horizons)    # 4

        # compress sat spatial context to single vector
        self.sat_pool = nn.Sequential(
            nn.Linear(D, D // 2),
            nn.GELU()
        )

        # temp + pooled_sat -> severity per horizon
        # input: temp(128) + sat_pooled(64) = 192
        self.forecast = nn.Sequential(
            nn.Linear(D + D // 2, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, H)    # one output per horizon
        )

        # horizon-specific scaling -- each horizon has different typical range
        self.horizon_scale = nn.Parameter(torch.ones(H))
        self.horizons = horizons

    def forward(self, temp_token, sat_tokens):
        # temp_token : (B, 1, D)
        # sat_tokens : (B, 64, D)

        temp = temp_token.squeeze(1)                # (B, D)

        # mean pool sat tokens for global spatial context
        sat_ctx = self.sat_pool(sat_tokens.mean(1)) # (B, D//2)

        combined = torch.cat([temp, sat_ctx], dim=-1)  # (B, D + D//2)
        severity = self.forecast(combined)              # (B, H)

        # sigmoid -> 0-1 severity scale per horizon
        # multiply by learned scale per horizon
        severity = torch.sigmoid(severity) * self.horizon_scale.abs()

        return severity   # (B, 4)


# ── test ──────────────────────────────────────────────────────
sp2 = SeverityForecastHead(
    fusion_dim=config.model.fusion_dim,
    horizons=config.data.forecast_horizons
).cuda()

temp_token = torch.randn(2, 1, 128).cuda()
sat_tokens = torch.randn(2, 64, 128).cuda()

with torch.no_grad():
    severity = sp2(temp_token, sat_tokens)

print("SP-2 SeverityForecastHead:")
print(f"  temp input : {temp_token.shape}")
print(f"  sat input  : {sat_tokens.shape}")
print(f"  output     : {severity.shape}  <-- (2, 4) one per horizon")
print(f"  horizons   : {config.data.forecast_horizons}h")
print(f"  severity   : {severity[0].tolist()}")
print(f"  range      : [{severity.min():.3f}, {severity.max():.3f}]  <-- must be > 0")
print(f"  params     : {sum(p.numel() for p in sp2.parameters()):,}")
assert severity.shape == (2, 4)
print("  PASSED")

SP-2 SeverityForecastHead:
  temp input : torch.Size([2, 1, 128])
  sat input  : torch.Size([2, 64, 128])
  output     : torch.Size([2, 4])  <-- (2, 4) one per horizon
  horizons   : [6, 24, 48, 72]h
  severity   : [0.4629760682582855, 0.49097198247909546, 0.5052114129066467, 0.5075649619102478]
  range      : [0.463, 0.511]  <-- must be > 0
  params     : 91,080
  PASSED


In [7]:
# SP-3: risk classification per region
# 4 classes: low / medium / high / critical
# input: all three fused modalities

# SP-4: infrastructure damage assessment
# 3 classes: none / partial / severe
# input: sat (spatial detail) + static (knows where infra is)

class RiskScorer(nn.Module):
    # SP-3

    def __init__(self, fusion_dim, n_classes=4):
        super().__init__()
        D = fusion_dim

        # all three modalities contribute to risk
        self.sat_proj  = nn.Linear(D, D // 2)
        self.temp_proj = nn.Linear(D, D // 2)
        self.stat_proj = nn.Linear(D, D // 2)

        combined_dim = (D // 2) * 3   # 192

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, n_classes)
            # no softmax -- CrossEntropyLoss handles it
        )

    def forward(self, sat, temp, stat):
        # pool each to (B, D//2)
        s  = self.sat_proj(sat.mean(1))
        t  = self.temp_proj(temp.squeeze(1))
        st = self.stat_proj(stat.mean(1))

        x = torch.cat([s, t, st], dim=-1)   # (B, 192)
        return self.classifier(x)            # (B, 4) logits


class DamageAssessor(nn.Module):
    # SP-4
    # focuses on satellite (sees current state) + static (knows infra locations)
    # temporal less important here -- damage is instantaneous

    def __init__(self, fusion_dim, pool_size=8, n_classes=3):
        super().__init__()
        D = fusion_dim

        # pixel-level damage map (similar to SP-1 but 3-class)
        def up_block(ic, oc):
            return nn.Sequential(
                nn.ConvTranspose2d(ic, oc, 2, stride=2),
                nn.BatchNorm2d(oc), nn.ReLU(True),
                nn.Conv2d(oc, oc, 3, padding=1, bias=False),
                nn.BatchNorm2d(oc), nn.ReLU(True)
            )

        self.reshape = nn.Linear(D, D)
        self.up1  = up_block(D*2, 64)   # sat+stat concat -> up
        self.up2  = up_block(64, 32)
        self.up3  = up_block(32, 16)
        self.head = nn.Conv2d(16, n_classes, 1)
        self.pool_size = pool_size

    def forward(self, sat_tokens, stat_tokens):
        B = sat_tokens.shape[0]
        P = self.pool_size
        D = sat_tokens.shape[-1]

        # reshape both to spatial maps then concat
        s  = self.reshape(sat_tokens).transpose(1,2).reshape(B, D, P, P)
        st = self.reshape(stat_tokens).transpose(1,2).reshape(B, D, P, P)
        x  = torch.cat([s, st], dim=1)   # (B, D*2, 8, 8)

        x = self.up1(x)    # (B, 64,  16, 16)
        x = self.up2(x)    # (B, 32,  32, 32)
        x = self.up3(x)    # (B, 16,  64, 64)

        return self.head(x)   # (B, 3, 64, 64) -- 3-class damage map


# ── test both ─────────────────────────────────────────────────
sp3 = RiskScorer(config.model.fusion_dim).cuda()
sp4 = DamageAssessor(config.model.fusion_dim, config.model.fusion_pool_size).cuda()

sat_t  = torch.randn(2, 64, 128).cuda()
temp_t = torch.randn(2, 1,  128).cuda()
stat_t = torch.randn(2, 64, 128).cuda()

with torch.no_grad():
    risk   = sp3(sat_t, temp_t, stat_t)
    damage = sp4(sat_t, stat_t)

print("SP-3 RiskScorer:")
print(f"  output : {risk.shape}   <-- (2, 4) risk class logits")
print(f"  params : {sum(p.numel() for p in sp3.parameters()):,}")
assert risk.shape == (2, 4)
print("  PASSED")

print("\nSP-4 DamageAssessor:")
print(f"  output : {damage.shape}  <-- (2, 3, 64, 64) damage map")
print(f"  params : {sum(p.numel() for p in sp4.parameters()):,}")
assert damage.shape == (2, 3, 64, 64)
print("  PASSED")

SP-3 RiskScorer:
  output : torch.Size([2, 4])   <-- (2, 4) risk class logits
  params : 90,884
  PASSED

SP-4 DamageAssessor:
  output : torch.Size([2, 3, 64, 64])  <-- (2, 3, 64, 64) damage map
  params : 141,283
  PASSED


In [8]:
# single forward pass through everything built so far
# verifies shapes are compatible end to end

torch.cuda.empty_cache()

# rebuild all
encoder    = SpatioTemporalEncoder(config).cuda()
temp_enc   = TemporalEncoder(config).cuda()
static_enc = StaticMapEncoder(config).cuda()
fusion     = CrossModalFusion(config).cuda()
sp1        = FloodMapDecoder(config.model.fusion_dim, config.model.fusion_pool_size).cuda()
sp2        = SeverityForecastHead(config.model.fusion_dim, config.data.forecast_horizons).cuda()
sp3        = RiskScorer(config.model.fusion_dim).cuda()
sp4        = DamageAssessor(config.model.fusion_dim, config.model.fusion_pool_size).cuda()

B = 2
sat_in  = torch.randn(B, 12, 4, 64, 64).cuda()
w_in    = torch.randn(B, 72,  7).cuda()
g_in    = torch.randn(B, 72,  3).cuda()
st_in   = torch.randn(B,  5, 64, 64).cuda()

torch.cuda.empty_cache()
m0 = torch.cuda.memory_allocated(0)/1e9

with torch.amp.autocast('cuda'):
    sat_feat           = encoder(sat_in)
    temp_feat, _       = temp_enc(w_in, g_in)
    stat_feat, _       = static_enc(st_in)
    so, to, sto, (H,W) = fusion(sat_feat, temp_feat, stat_feat)

    flood_logits, uncert = sp1(so)
    severity             = sp2(to, so)
    risk                 = sp3(so, to, sto)
    damage               = sp4(so, sto)

m1 = torch.cuda.memory_allocated(0)/1e9

print("Full pipeline (encoders + fusion + SP-1..4):")
print(f"  flood map  : {flood_logits.shape}")
print(f"  uncertainty: {uncert.shape}")
print(f"  severity   : {severity.shape}  (6h/24h/48h/72h)")
print(f"  risk class : {risk.shape}      (low/med/high/critical)")
print(f"  damage map : {damage.shape}")
print(f"  memory     : {m1:.2f} GB  |  remaining: {15.6-m1:.2f} GB")

total = sum(
    sum(p.numel() for p in m.parameters())
    for m in [encoder,temp_enc,static_enc,fusion,sp1,sp2,sp3,sp4]
)
print(f"  total params: {total:,}")
print("\n  SP-1 to SP-4 -- all connected end to end")

Full pipeline (encoders + fusion + SP-1..4):
  flood map  : torch.Size([2, 1, 64, 64])
  uncertainty: torch.Size([2, 1, 64, 64])
  severity   : torch.Size([2, 4])  (6h/24h/48h/72h)
  risk class : torch.Size([2, 4])      (low/med/high/critical)
  damage map : torch.Size([2, 3, 64, 64])
  memory     : 0.56 GB  |  remaining: 15.04 GB
  total params: 4,205,531

  SP-1 to SP-4 -- all connected end to end


/tmp/ipykernel_24/1436218676.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, depth)


In [9]:
# SP-5: given flood map + damage map, find safest evacuation routes
# roads = graph nodes, connections = edges
# flood probability on each road segment = node feature
# GNN propagates danger information through road network

class GraphAttentionLayer(nn.Module):
    # single GAT layer -- each node attends to its neighbors
    # learns: "how dangerous is my neighbor? should I route through them?"

    def __init__(self, in_dim, out_dim, heads=4, dropout=0.1):
        super().__init__()
        self.heads   = heads
        self.out_dim = out_dim
        head_dim     = out_dim // heads

        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Linear(2 * head_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.act  = nn.ELU()
        self._head_dim = head_dim

    def forward(self, x, adj):
        # x  : (B, N, in_dim)  -- N road segments
        # adj: (B, N, N)       -- adjacency matrix (1 if road connected)
        B, N, _ = x.shape
        H        = self.heads
        D        = self._head_dim

        x_proj = self.W(x).reshape(B, N, H, D)     # (B, N, H, D)

        # pairwise attention: concat each node with each neighbor
        xi = x_proj.unsqueeze(2).expand(-1,-1,N,-1,-1)  # (B, N, N, H, D)
        xj = x_proj.unsqueeze(1).expand(-1,N,-1,-1,-1)  # (B, N, N, H, D)
        e  = self.attn(torch.cat([xi, xj], dim=-1)).squeeze(-1)  # (B, N, N, H)

        # mask non-edges with -inf before softmax
        mask = (adj == 0).unsqueeze(-1).expand_as(e)
        e    = e.masked_fill(mask, -1e9)
        a    = torch.softmax(e, dim=2)               # (B, N, N, H)
        a    = self.drop(a)

        # aggregate neighbor features
        out = (a.unsqueeze(-1) * xj).sum(dim=2)     # (B, N, H, D)
        out = out.reshape(B, N, H * D)               # (B, N, out_dim)
        return self.act(out)


class EvacuationGNN(nn.Module):
    # SP-5
    # input : flood logits (SP-1 output) projected to graph nodes
    # output: danger score per road segment + recommended route weights

    def __init__(self, fusion_dim, gnn_hidden, n_layers, n_heads, n_nodes=64):
        super().__init__()
        # n_nodes = number of road segment nodes in graph
        # we use 64 to match our 8x8 spatial grid -- one node per grid cell
        self.n_nodes = n_nodes

        # project flood + static features to node features
        self.node_proj = nn.Sequential(
            nn.Linear(fusion_dim + 1, gnn_hidden),  # +1 for flood probability
            nn.LayerNorm(gnn_hidden),
            nn.ReLU(True)
        )

        # stack of GAT layers
        self.gat_layers = nn.ModuleList([
            GraphAttentionLayer(gnn_hidden, gnn_hidden, n_heads)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(gnn_hidden) for _ in range(n_layers)])

        # output: danger score per node (higher = avoid this road)
        self.danger_head = nn.Sequential(
            nn.Linear(gnn_hidden, 32),
            nn.ReLU(True),
            nn.Linear(32, 1),
            nn.Sigmoid()    # 0 = safe, 1 = dangerous
        )

        # route recommendation: which nodes are on optimal evac routes
        self.route_head = nn.Sequential(
            nn.Linear(gnn_hidden, 32),
            nn.ReLU(True),
            nn.Linear(32, 1),
            nn.Sigmoid()    # probability this node is on safe route
        )

    def _build_grid_adjacency(self, B, N, device):
        # build adjacency for NxN grid graph
        # each cell connected to its 4 neighbors (up/down/left/right)
        size = int(N ** 0.5)   # 8 for N=64
        adj  = torch.zeros(B, N, N, device=device)
        for i in range(N):
            r, c = i // size, i % size
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < size and 0 <= nc < size:
                    j = nr * size + nc
                    adj[:, i, j] = 1.0
        return adj

    def forward(self, sat_tokens, flood_logits):
        # sat_tokens  : (B, 64, D)  -- node features from fusion
        # flood_logits: (B, 1, 64, 64) -- SP-1 output

        B, N, D = sat_tokens.shape

        # project flood map to node space (pool 64x64 -> 8x8 -> flatten)
        flood_node = F.adaptive_avg_pool2d(
            flood_logits.sigmoid(), (8, 8)
        ).flatten(2).transpose(1, 2)           # (B, 64, 1)

        # combine fusion features + flood probability per node
        node_feat = torch.cat([sat_tokens, flood_node], dim=-1)  # (B, 64, D+1)
        x         = self.node_proj(node_feat)                     # (B, 64, gnn_hidden)

        # build adjacency matrix
        adj = self._build_grid_adjacency(B, N, x.device)

        # propagate through GAT layers
        for gat, norm in zip(self.gat_layers, self.norms):
            x = norm(x + gat(x, adj))   # residual + norm

        danger = self.danger_head(x)    # (B, 64, 1)
        routes = self.route_head(x)     # (B, 64, 1)

        # reshape to spatial maps for visualization
        danger_map = danger.transpose(1,2).reshape(B, 1, 8, 8)
        route_map  = routes.transpose(1,2).reshape(B, 1, 8, 8)

        return danger_map, route_map    # both (B, 1, 8, 8)


# ── test ──────────────────────────────────────────────────────
sp5 = EvacuationGNN(
    fusion_dim=config.model.fusion_dim,
    gnn_hidden=config.model.gnn_hidden_dim,
    n_layers=config.model.gnn_layers,
    n_heads=config.model.gnn_heads,
    n_nodes=64
).cuda()

sat_t        = torch.randn(2, 64, 128).cuda()
flood_logits = torch.randn(2, 1, 64, 64).cuda()

with torch.no_grad():
    danger_map, route_map = sp5(sat_t, flood_logits)

print("SP-5 EvacuationGNN:")
print(f"  danger map : {danger_map.shape}  <-- (2, 1, 8, 8)")
print(f"  route map  : {route_map.shape}   <-- (2, 1, 8, 8)")
print(f"  danger range: [{danger_map.min():.3f}, {danger_map.max():.3f}]  <-- 0-1")
print(f"  route range : [{route_map.min():.3f}, {route_map.max():.3f}]   <-- 0-1")
print(f"  params : {sum(p.numel() for p in sp5.parameters()):,}")
assert danger_map.shape == (2, 1, 8, 8)
assert danger_map.min() >= 0 and danger_map.max() <= 1
print("  PASSED")

SP-5 EvacuationGNN:
  danger map : torch.Size([2, 1, 8, 8])  <-- (2, 1, 8, 8)
  route map  : torch.Size([2, 1, 8, 8])   <-- (2, 1, 8, 8)
  danger range: [0.306, 0.543]  <-- 0-1
  route range : [0.368, 0.663]   <-- 0-1
  params : 25,442
  PASSED


In [10]:
class ResourceAllocationPolicy(nn.Module):

    def __init__(self, fusion_dim, n_nodes, action_dim, rl_state_dim):
        super().__init__()
        D = fusion_dim

        # correct dim: sat(128) + risk(n_nodes*4) + damage(n_nodes*3) + danger(n_nodes)
        state_dim = D + n_nodes*4 + n_nodes*3 + n_nodes   # 128+256+192+64 = 640

        self.state_encoder = nn.Sequential(
            nn.Linear(state_dim, rl_state_dim),
            nn.LayerNorm(rl_state_dim),
            nn.GELU(),
            nn.Linear(rl_state_dim, rl_state_dim),
            nn.GELU()
        )
        self.actor = nn.Sequential(
            nn.Linear(rl_state_dim, 128),
            nn.GELU(),
            nn.Linear(128, action_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(rl_state_dim, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )
        self.resource_types = ["ambulances","rescue_teams","helicopters","supply_trucks"]

    def forward(self, sat_tokens, risk_logits, damage_logits, danger_map):
        B = sat_tokens.shape[0]

        sat_ctx   = sat_tokens.mean(1)                                          # (B, 128)
        risk_rep  = risk_logits.unsqueeze(1).expand(-1,64,-1).reshape(B,-1)    # (B, 256)
        dmg_nodes = F.adaptive_avg_pool2d(damage_logits.float(),(8,8)).flatten(1) # (B, 192)
        dng_nodes = danger_map.flatten(1)                                       # (B, 64)

        state   = torch.cat([sat_ctx, risk_rep, dmg_nodes, dng_nodes], dim=-1) # (B, 640)
        encoded = self.state_encoder(state)

        action_logits = self.actor(encoded)
        value         = self.critic(encoded)
        action_probs  = torch.softmax(action_logits, dim=-1)

        return action_probs, value, encoded


# ── rebuild + test ────────────────────────────────────────────
sp6 = ResourceAllocationPolicy(
    fusion_dim=config.model.fusion_dim,
    n_nodes=64,
    action_dim=config.model.rl_action_dim,
    rl_state_dim=config.model.rl_state_dim
).cuda()

sat_t    = torch.randn(2, 64, 128).cuda()
risk_t   = torch.randn(2, 4).cuda()
damage_t = torch.randn(2, 3, 64, 64).cuda()
danger_t = torch.randn(2, 1, 8, 8).cuda()

with torch.no_grad():
    probs, value, state_enc = sp6(sat_t, risk_t, damage_t, danger_t)

print("SP-6 ResourceAllocationPolicy:")
print(f"  action probs : {probs.shape}   <-- (2, 64)")
print(f"  value est    : {value.shape}       <-- (2, 1)")
print(f"  state enc    : {state_enc.shape}  <-- (2, 256)")
print(f"  probs sum    : {probs.sum(-1).tolist()}  <-- must be ~1.0")
print(f"  params       : {sum(p.numel() for p in sp6.parameters()):,}")
assert probs.shape == (2, config.model.rl_action_dim)
assert abs(probs[0].sum().item() - 1.0) < 1e-4
print("  PASSED")

SP-6 ResourceAllocationPolicy:
  action probs : torch.Size([2, 64])   <-- (2, 64)
  value est    : torch.Size([2, 1])       <-- (2, 1)
  state enc    : torch.Size([2, 256])  <-- (2, 256)
  probs sum    : [1.0, 1.0]  <-- must be ~1.0
  params       : 304,577
  PASSED


In [11]:
# SP-7: causal discovery -- what CAUSES floods in this region?
# SP-8: uncertainty quantification -- how confident is our prediction?
# SP-9: climate adaptation -- how will risk change by 2030/2040/2050?

class CausalDiscoveryHead(nn.Module):
    # SP-7
    # learns a causal adjacency matrix over input variables
    # based on NOTEARS: treats causal discovery as continuous optimization
    # output: which variables causally influence flood risk

    def __init__(self, n_vars, hidden_dim):
        super().__init__()
        self.n_vars = n_vars

        # encode each variable's temporal pattern
        self.var_encoder = nn.Linear(hidden_dim, hidden_dim)

        # causal adjacency: A[i,j] = "variable i causes variable j"
        # we learn this as a continuous matrix, then threshold
        self.causal_W = nn.Parameter(torch.randn(n_vars, n_vars) * 0.01)

        # DAG constraint projection -- keeps learned graph acyclic
        # using matrix exponential trick from NOTEARS paper
        self.proj = nn.Linear(n_vars, n_vars)

    def forward(self, temp_feat, sel_weights):
        # temp_feat   : (B, T, 64)  -- temporal encoder output
        # sel_weights : (B, T, 10)  -- variable importance weights

        # average variable importance over time -> (B, 10)
        avg_importance = sel_weights.mean(dim=1)   # (B, n_vars)

        # soft causal adjacency: sigmoid so values in 0-1
        # row i, col j: probability that var i -> var j
        causal_adj = torch.sigmoid(self.causal_W)

        # mask diagonal (no self-causation)
        mask       = 1 - torch.eye(self.n_vars, device=causal_adj.device)
        causal_adj = causal_adj * mask

        # apply to variable importances -> causal influence scores
        causal_influence = avg_importance @ causal_adj  # (B, n_vars)

        return causal_adj, causal_influence


class UncertaintyHead(nn.Module):
    # SP-8: epistemic uncertainty via MC Dropout
    # during inference: run N forward passes with dropout ON
    # variance across passes = model uncertainty
    # here we define the dropout wrapper -- MC sampling in trainer

    def __init__(self, fusion_dim, n_samples=50):
        super().__init__()
        self.n_samples = n_samples
        # additional dropout layers for MC sampling
        self.drop1 = nn.Dropout(p=0.2)
        self.drop2 = nn.Dropout(p=0.2)
        self.proj  = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU()
        )
        # calibration head: learns to scale raw uncertainty estimates
        self.calibrate = nn.Sequential(
            nn.Linear(fusion_dim, 32),
            nn.ReLU(True),
            nn.Linear(32, 1),
            nn.Softplus()    # always positive
        )

    def forward(self, sat_tokens):
        # always apply dropout even at eval() -- MC Dropout trick
        # caller must set model.train() or use dropout manually
        x   = self.drop1(sat_tokens)
        x   = self.proj(x)
        x   = self.drop2(x)
        unc = self.calibrate(x.mean(1))  # (B, 1) -- global uncertainty
        return unc

    def mc_sample(self, sat_tokens, n_samples=None):
        # run multiple stochastic forward passes
        # returns mean + variance across samples
        n = n_samples or self.n_samples
        self.train()   # ensure dropout is active
        samples = torch.stack([self.forward(sat_tokens) for _ in range(n)], dim=0)
        # samples: (n_samples, B, 1)
        return samples.mean(0), samples.var(0)   # mean_unc, var_unc


class ClimateAdaptationHead(nn.Module):
    # SP-9: given current risk + long-term temporal patterns,
    # project flood vulnerability to future years

    def __init__(self, fusion_dim, climate_years):
        super().__init__()
        D = fusion_dim
        n_years = len(climate_years)

        # takes current situation + temporal context
        self.encoder = nn.Sequential(
            nn.Linear(D * 2, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.GELU()
        )

        # one output per future year: vulnerability score 0-1
        self.projector = nn.Linear(128, n_years)

        # trend uncertainty -- how confident are we in long-term projection?
        self.trend_uncert = nn.Sequential(
            nn.Linear(128, n_years),
            nn.Softplus()
        )

        self.years = climate_years

    def forward(self, sat_tokens, temp_token):
        sat_ctx  = sat_tokens.mean(1)       # (B, D)
        temp_ctx = temp_token.squeeze(1)    # (B, D)

        x = torch.cat([sat_ctx, temp_ctx], dim=-1)  # (B, D*2)
        h = self.encoder(x)                          # (B, 128)

        # vulnerability per future year (sigmoid -> 0-1)
        vulnerability = torch.sigmoid(self.projector(h))   # (B, n_years)
        uncertainty   = self.trend_uncert(h)               # (B, n_years)

        return vulnerability, uncertainty


# ── test all three ────────────────────────────────────────────
n_vars = len(config.data.weather_features) + len(config.data.gauge_features)

sp7 = CausalDiscoveryHead(n_vars=n_vars, hidden_dim=config.model.tft_hidden_dim).cuda()
sp8 = UncertaintyHead(fusion_dim=config.model.fusion_dim).cuda()
sp9 = ClimateAdaptationHead(config.model.fusion_dim, config.data.forecast_horizons).cuda()

sat_t  = torch.randn(2, 64, 128).cuda()
temp_t = torch.randn(2, 1,  128).cuda()
tf_t   = torch.randn(2, 72,  64).cuda()   # temporal encoder output
sw_t   = torch.randn(2, 72,  10).cuda()   # selection weights

with torch.no_grad():
    causal_adj, causal_inf = sp7(tf_t, sw_t)
    mc_mean, mc_var        = sp8.mc_sample(sat_t, n_samples=10)
    vuln, trend_unc        = sp9(sat_t, temp_t)

print("SP-7 CausalDiscoveryHead:")
print(f"  causal adj    : {causal_adj.shape}   <-- (10, 10) var->var causality")
print(f"  causal inf    : {causal_inf.shape}  <-- (2, 10) per-variable influence")
print(f"  adj diagonal  : {causal_adj.diagonal().mean():.4f}  <-- must be 0")
print(f"  params        : {sum(p.numel() for p in sp7.parameters()):,}")
assert causal_adj.shape == (n_vars, n_vars)
assert causal_adj.diagonal().sum() < 1e-6
print("  PASSED")

print("\nSP-8 UncertaintyHead (MC Dropout, 10 samples):")
print(f"  mc_mean  : {mc_mean.shape}  <-- (2, 1)")
print(f"  mc_var   : {mc_var.shape}   <-- (2, 1) epistemic uncertainty")
print(f"  mean unc : {mc_mean.mean():.4f}")
print(f"  params   : {sum(p.numel() for p in sp8.parameters()):,}")
print("  PASSED")

print("\nSP-9 ClimateAdaptationHead:")
print(f"  vulnerability : {vuln.shape}      <-- (2, 4) one per year")
print(f"  trend uncert  : {trend_unc.shape} <-- (2, 4)")
print(f"  vuln range    : [{vuln.min():.3f}, {vuln.max():.3f}]  <-- 0-1")
print(f"  years         : {config.data.forecast_horizons}")
print(f"  params        : {sum(p.numel() for p in sp9.parameters()):,}")
assert vuln.shape == (2, len(config.data.forecast_horizons))
print("  PASSED")

SP-7 CausalDiscoveryHead:
  causal adj    : torch.Size([10, 10])   <-- (10, 10) var->var causality
  causal inf    : torch.Size([2, 10])  <-- (2, 10) per-variable influence
  adj diagonal  : 0.0000  <-- must be 0
  params        : 4,370
  PASSED

SP-8 UncertaintyHead (MC Dropout, 10 samples):
  mc_mean  : torch.Size([2, 1])  <-- (2, 1)
  mc_var   : torch.Size([2, 1])   <-- (2, 1) epistemic uncertainty
  mean unc : 0.7318
  params   : 20,673
  PASSED

SP-9 ClimateAdaptationHead:
  vulnerability : torch.Size([2, 4])      <-- (2, 4) one per year
  trend uncert  : torch.Size([2, 4]) <-- (2, 4)
  vuln range    : [0.481, 0.508]  <-- 0-1
  years         : [6, 24, 48, 72]
  params        : 99,720
  PASSED


In [12]:
# fixed GraphAttentionLayer -- fp16 safe
class GraphAttentionLayer(nn.Module):

    def __init__(self, in_dim, out_dim, heads=4, dropout=0.1):
        super().__init__()
        self.heads     = heads
        self.out_dim   = out_dim
        head_dim       = out_dim // heads
        self.W         = nn.Linear(in_dim, out_dim, bias=False)
        self.attn      = nn.Linear(2 * head_dim, 1, bias=False)
        self.drop      = nn.Dropout(dropout)
        self.act       = nn.ELU()
        self._head_dim = head_dim

    def forward(self, x, adj):
        B, N, _ = x.shape
        H, D    = self.heads, self._head_dim

        x_proj = self.W(x).reshape(B, N, H, D)
        xi     = x_proj.unsqueeze(2).expand(-1,-1,N,-1,-1)
        xj     = x_proj.unsqueeze(1).expand(-1,N,-1,-1,-1)
        e      = self.attn(torch.cat([xi, xj], dim=-1)).squeeze(-1)

        mask = (adj == 0).unsqueeze(-1).expand_as(e)
        e    = e.masked_fill(mask, -1e4)    # -1e4 not -1e9 -- fp16 safe
        a    = torch.softmax(e, dim=2)
        a    = self.drop(a)

        out = (a.unsqueeze(-1) * xj).sum(dim=2)
        return self.act(out.reshape(B, N, H * D))


class EvacuationGNN(nn.Module):

    def __init__(self, fusion_dim, gnn_hidden, n_layers, n_heads, n_nodes=64):
        super().__init__()
        self.n_nodes  = n_nodes
        self.node_proj = nn.Sequential(
            nn.Linear(fusion_dim + 1, gnn_hidden),
            nn.LayerNorm(gnn_hidden),
            nn.ReLU(True)
        )
        self.gat_layers = nn.ModuleList([
            GraphAttentionLayer(gnn_hidden, gnn_hidden, n_heads)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(gnn_hidden) for _ in range(n_layers)
        ])
        self.danger_head = nn.Sequential(
            nn.Linear(gnn_hidden, 32), nn.ReLU(True),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        self.route_head = nn.Sequential(
            nn.Linear(gnn_hidden, 32), nn.ReLU(True),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def _build_grid_adjacency(self, B, N, device):
        size = int(N ** 0.5)
        adj  = torch.zeros(B, N, N, device=device)
        for i in range(N):
            r, c = i // size, i % size
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < size and 0 <= nc < size:
                    adj[:, i, nr*size+nc] = 1.0
        return adj

    def forward(self, sat_tokens, flood_logits):
        B, N, D = sat_tokens.shape
        flood_node = F.adaptive_avg_pool2d(
            flood_logits.sigmoid(), (8,8)
        ).flatten(2).transpose(1,2)                     # (B, 64, 1)

        node_feat = torch.cat([sat_tokens, flood_node], dim=-1)
        x         = self.node_proj(node_feat)
        adj       = self._build_grid_adjacency(B, N, x.device)

        for gat, norm in zip(self.gat_layers, self.norms):
            x = norm(x + gat(x, adj))

        danger = self.danger_head(x).transpose(1,2).reshape(B,1,8,8)
        routes = self.route_head(x).transpose(1,2).reshape(B,1,8,8)
        return danger, routes


class FloodCastNet(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        D      = cfg.model.fusion_dim
        P      = cfg.model.fusion_pool_size
        n_vars = len(cfg.data.weather_features) + len(cfg.data.gauge_features)

        self.sat_encoder  = SpatioTemporalEncoder(cfg)
        self.temp_encoder = TemporalEncoder(cfg)
        self.stat_encoder = StaticMapEncoder(cfg)
        self.fusion       = CrossModalFusion(cfg)

        self.sp1 = FloodMapDecoder(D, P)
        self.sp2 = SeverityForecastHead(D, cfg.data.forecast_horizons)
        self.sp3 = RiskScorer(D)
        self.sp4 = DamageAssessor(D, P)
        self.sp5 = EvacuationGNN(D, cfg.model.gnn_hidden_dim,
                                  cfg.model.gnn_layers, cfg.model.gnn_heads)
        self.sp6 = ResourceAllocationPolicy(D, 64,
                                            cfg.model.rl_action_dim,
                                            cfg.model.rl_state_dim)
        self.sp7 = CausalDiscoveryHead(n_vars, cfg.model.tft_hidden_dim)
        self.sp8 = UncertaintyHead(D, cfg.model.mc_dropout_samples)
        self.sp9 = ClimateAdaptationHead(D, cfg.data.forecast_horizons)

    def forward(self, sat, weather, gauge, static_maps):
        sat_feat            = self.sat_encoder(sat)
        temp_feat, sel_w    = self.temp_encoder(weather, gauge)
        stat_feat, _        = self.stat_encoder(static_maps)

        so, to, sto, _      = self.fusion(sat_feat, temp_feat, stat_feat)

        flood_logits, aleat_unc         = self.sp1(so)
        severity                        = self.sp2(to, so)
        risk                            = self.sp3(so, to, sto)
        damage                          = self.sp4(so, sto)
        danger_map, route_map           = self.sp5(so, flood_logits)
        action_probs, value, _          = self.sp6(so, risk, damage, danger_map)
        causal_adj, causal_inf          = self.sp7(temp_feat, sel_w)
        epistemic_unc                   = self.sp8(so)
        vulnerability, cl_unc           = self.sp9(so, to)

        return {
            "flood_map"     : flood_logits,
            "aleatoric_unc" : aleat_unc,
            "severity"      : severity,
            "risk"          : risk,
            "damage_map"    : damage,
            "danger_map"    : danger_map,
            "route_map"     : route_map,
            "action_probs"  : action_probs,
            "rl_value"      : value,
            "causal_adj"    : causal_adj,
            "causal_inf"    : causal_inf,
            "epistemic_unc" : epistemic_unc,
            "vulnerability" : vulnerability,
            "climate_unc"   : cl_unc,
        }


# ── final assembly test ───────────────────────────────────────
torch.cuda.empty_cache()
model = FloodCastNet(config).cuda()

B      = 2
inputs = {
    "sat"         : torch.randn(B, 12,  4, 64, 64).cuda(),
    "weather"     : torch.randn(B, 72,  7).cuda(),
    "gauge"       : torch.randn(B, 72,  3).cuda(),
    "static_maps" : torch.randn(B,  5, 64, 64).cuda()
}

with torch.amp.autocast('cuda'):
    out = model(**inputs)

print("FloodCastNet — Complete Forward Pass:")
print("=" * 50)
for k, v in out.items():
    print(f"  {k:<18}: {str(v.shape)}")

total = sum(p.numel() for p in model.parameters())
mem   = torch.cuda.memory_allocated(0) / 1e9
print("=" * 50)
print(f"  Total params  : {total:,}")
print(f"  GPU memory    : {mem:.2f} GB  |  remaining: {15.6-mem:.2f} GB")
print("\n  FloodCastNet complete -- all 9 sub-problems connected")

FloodCastNet — Complete Forward Pass:
  flood_map         : torch.Size([2, 1, 64, 64])
  aleatoric_unc     : torch.Size([2, 1, 64, 64])
  severity          : torch.Size([2, 4])
  risk              : torch.Size([2, 4])
  damage_map        : torch.Size([2, 3, 64, 64])
  danger_map        : torch.Size([2, 1, 8, 8])
  route_map         : torch.Size([2, 1, 8, 8])
  action_probs      : torch.Size([2, 64])
  rl_value          : torch.Size([2, 1])
  causal_adj        : torch.Size([10, 10])
  causal_inf        : torch.Size([2, 10])
  epistemic_unc     : torch.Size([2, 1])
  vulnerability     : torch.Size([2, 4])
  climate_unc       : torch.Size([2, 4])
  Total params  : 4,660,313
  GPU memory    : 1.09 GB  |  remaining: 14.51 GB

  FloodCastNet complete -- all 9 sub-problems connected


/tmp/ipykernel_24/1436218676.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, depth)


In [13]:
class PhysicsLoss(nn.Module):
    # water flows downhill -- flood predictions must respect elevation.
    # if model predicts high flood probability at high elevation
    # while ignoring low elevation neighbors, penalize it.
    # this is what makes FloodCastNet physically consistent.

    def __init__(self, weight=0.1):
        super().__init__()
        self.w = weight

        # sobel filters for elevation gradient
        sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('sx', sx)
        self.register_buffer('sy', sy)

    def forward(self, flood_logits, dem):
        # flood_logits: (B, 1, H, W) -- raw logits from SP-1
        # dem         : (B, 1, H, W) -- elevation map (from static input channel 0)

        flood_prob = torch.sigmoid(flood_logits)

        # elevation gradient -- direction water would flow
        grad_ex = F.conv2d(dem.float(),        self.sx, padding=1)
        grad_ey = F.conv2d(dem.float(),        self.sy, padding=1)

        # flood probability gradient
        grad_fx = F.conv2d(flood_prob.float(), self.sx, padding=1)
        grad_fy = F.conv2d(flood_prob.float(), self.sy, padding=1)

        # violation: flood increasing in direction of increasing elevation
        # relu keeps only positive violations (actual physics breaks)
        viol_x = F.relu(grad_fx * grad_ex)
        viol_y = F.relu(grad_fy * grad_ey)

        return self.w * (viol_x + viol_y).mean()


class MultiTaskLoss(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.weights  = cfg.training.loss_weights
        self.physics  = PhysicsLoss(weight=cfg.training.loss_weights["physics"])

        # class weights for flood -- heavily imbalanced (few flood pixels)
        self.bce_flood  = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([10.0])   # flood pixels ~10x rarer
        )
        self.ce_risk    = nn.CrossEntropyLoss()
        self.ce_damage  = nn.CrossEntropyLoss()
        self.mse_sev    = nn.MSELoss()

    def forward(self, preds, targets, static_maps):
        # static_maps[:,0] = DEM -- used for physics loss
        dem = static_maps[:, 0:1]   # (B, 1, H, W)

        losses = {}

        # SP-1: flood map segmentation
        if "flood_mask" in targets:
            losses["sp1"] = self.weights["sp1_flood_map"] * \
                self.bce_flood(preds["flood_map"], targets["flood_mask"])

        # SP-1 physics constraint
        losses["physics"] = self.physics(preds["flood_map"], dem)

        # SP-2: severity forecasting
        if "severity" in targets:
            losses["sp2"] = self.weights["sp2_severity"] * \
                self.mse_sev(preds["severity"], targets["severity"])

        # SP-3: risk classification
        if "risk" in targets:
            losses["sp3"] = self.weights["sp3_risk"] * \
                self.ce_risk(preds["risk"], targets["risk"])

        # SP-4: damage map
        if "damage_mask" in targets:
            # damage_map is (B, 3, H, W), target is (B, H, W) long
            losses["sp4"] = self.weights["sp4_damage"] * \
                self.ce_damage(preds["damage_map"], targets["damage_mask"])

        # SP-8: uncertainty calibration
        # predicted uncertainty should correlate with actual error
        if "flood_mask" in targets:
            with torch.no_grad():
                actual_error = (
                    torch.sigmoid(preds["flood_map"]) - targets["flood_mask"]
                ).abs().mean(dim=[2,3], keepdim=True)
            # aleatoric uncertainty should match actual pixel-level error
            unc_loss = F.mse_loss(preds["aleatoric_unc"], actual_error)
            losses["sp8"] = self.weights["sp8_uncertainty"] * unc_loss

        total = sum(losses.values())
        return total, losses


# ── test ──────────────────────────────────────────────────────
criterion = MultiTaskLoss(config).cuda()

# fake targets -- same shapes as real labels
B, H, W = 2, 64, 64
targets = {
    "flood_mask"  : torch.randint(0, 2, (B, 1, H, W)).float().cuda(),
    "severity"    : torch.rand(B, 4).cuda(),
    "risk"        : torch.randint(0, 4, (B,)).cuda(),
    "damage_mask" : torch.randint(0, 3, (B, H, W)).cuda()
}
static_in = torch.randn(B, 5, H, W).cuda()

with torch.amp.autocast('cuda'):
    out      = model(**{
        "sat"         : torch.randn(B,12,4,H,W).cuda(),
        "weather"     : torch.randn(B,72,7).cuda(),
        "gauge"       : torch.randn(B,72,3).cuda(),
        "static_maps" : static_in
    })
    total_loss, loss_dict = criterion(out, targets, static_in)

print("MultiTaskLoss test:")
print(f"  total loss : {total_loss.item():.4f}")
print()
for k, v in loss_dict.items():
    print(f"  {k:<10}: {v.item():.4f}")

print(f"\n  PASSED -- loss is differentiable and finite")
assert not torch.isnan(total_loss), "NaN loss"
assert not torch.isinf(total_loss), "Inf loss"

MultiTaskLoss test:
  total loss : 6.3550

  sp1       : 4.8163
  physics   : 0.0780
  sp2       : 0.0401
  sp3       : 0.8200
  sp4       : 0.5908
  sp8       : 0.0097

  PASSED -- loss is differentiable and finite


/tmp/ipykernel_24/1046107372.py:92: UserWarning: Using a target size (torch.Size([2, 1, 1, 1])) that is different to the input size (torch.Size([2, 1, 64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  unc_loss = F.mse_loss(preds["aleatoric_unc"], actual_error)


In [14]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import time

def train_one_epoch(model, optimizer, scheduler, scaler, criterion,
                    dataloader, cfg, epoch):

    model.train()
    total_loss   = 0.0
    loss_tracker = {}
    optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        sat         = batch["sat"].cuda()
        weather     = batch["weather"].cuda()
        gauge       = batch["gauge"].cuda()
        static_maps = batch["static_maps"].cuda()
        targets     = {k: v.cuda() for k, v in batch["targets"].items()}

        with torch.amp.autocast('cuda'):
            preds             = model(sat, weather, gauge, static_maps)
            loss, loss_dict   = criterion(preds, targets, static_maps)
            loss              = loss / cfg.training.gradient_accumulation_steps

        scaler.scale(loss).backward()

        # accumulate then step
        if (step + 1) % cfg.training.gradient_accumulation_steps == 0:
            # clip gradients -- prevents explosion in early training
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * cfg.training.gradient_accumulation_steps

        for k, v in loss_dict.items():
            loss_tracker[k] = loss_tracker.get(k, 0.0) + v.item()

        if step % 10 == 0:
            mem = torch.cuda.memory_allocated(0) / 1e9
            print(f"  epoch {epoch} step {step:03d} "
                  f"loss={total_loss/(step+1):.4f} "
                  f"mem={mem:.1f}GB")

    n = len(dataloader)
    return total_loss / n, {k: v/n for k, v in loss_tracker.items()}


def save_checkpoint(model, optimizer, epoch, loss, cfg):
    path = f"{cfg.checkpoint_dir}/epoch_{epoch:03d}_loss_{loss:.4f}.pt"
    torch.save({
        "epoch"     : epoch,
        "loss"      : loss,
        "model"     : model.state_dict(),
        "optimizer" : optimizer.state_dict(),
        "config"    : asdict(cfg)
    }, path)
    print(f"  checkpoint saved: {path}")
    return path


def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location='cuda')
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    print(f"  resumed from epoch {ckpt['epoch']}, loss {ckpt['loss']:.4f}")
    return ckpt["epoch"]


print("Training utilities defined")
print(f"  gradient accumulation : {config.training.gradient_accumulation_steps}")
print(f"  effective batch size  : {config.training.batch_size * config.training.gradient_accumulation_steps}")
print(f"  grad clip             : 1.0")
print(f"  AMP                   : enabled")

Training utilities defined
  gradient accumulation : 16
  effective batch size  : 32
  grad clip             : 1.0
  AMP                   : enabled


In [15]:
# we dont have real flood data yet -- build synthetic loader
# same shapes as real data -- lets us verify training loop works
# swap this out when real datasets are downloaded

class SyntheticFloodDataset(torch.utils.data.Dataset):

    def __init__(self, n_samples, cfg):
        self.n  = n_samples
        self.cfg = cfg

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        cfg = self.cfg
        H, W = 64, 64
        T    = cfg.data.sequence_length    # 12

        # satellite: (T, 4, H, W)
        sat = torch.randn(T, 4, H, W)

        # weather + gauge: (72, features)
        weather = torch.randn(cfg.data.weather_lookback, 7)
        gauge   = torch.randn(cfg.data.weather_lookback, 3)

        # static: (5, H, W) -- channel 0 is DEM
        static_maps = torch.randn(5, H, W)

        # targets
        # flood mask: sparse -- ~10% pixels flooded (realistic)
        flood_mask = (torch.rand(1, H, W) < 0.1).float()

        # severity: random per horizon
        severity = torch.rand(4)

        # risk class: 0-3
        risk = torch.randint(0, 4, ())

        # damage: 3-class per pixel
        damage_mask = torch.randint(0, 3, (H, W))

        return {
            "sat"         : sat,
            "weather"     : weather,
            "gauge"       : gauge,
            "static_maps" : static_maps,
            "targets"     : {
                "flood_mask"  : flood_mask,
                "severity"    : severity,
                "risk"        : risk.long(),
                "damage_mask" : damage_mask.long()
            }
        }


def collate_fn(batch):
    out = {
        "sat"         : torch.stack([b["sat"]         for b in batch]),
        "weather"     : torch.stack([b["weather"]     for b in batch]),
        "gauge"       : torch.stack([b["gauge"]       for b in batch]),
        "static_maps" : torch.stack([b["static_maps"] for b in batch]),
        "targets"     : {
            k: torch.stack([b["targets"][k] for b in batch])
            for k in batch[0]["targets"]
        }
    }
    return out


dataset    = SyntheticFloodDataset(n_samples=64, cfg=config)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# verify one batch
batch = next(iter(dataloader))
print("DataLoader test:")
print(f"  sat         : {batch['sat'].shape}")
print(f"  weather     : {batch['weather'].shape}")
print(f"  gauge       : {batch['gauge'].shape}")
print(f"  static_maps : {batch['static_maps'].shape}")
print(f"  flood_mask  : {batch['targets']['flood_mask'].shape}")
print(f"  severity    : {batch['targets']['severity'].shape}")
print(f"  risk        : {batch['targets']['risk'].shape}")
print(f"  damage_mask : {batch['targets']['damage_mask'].shape}")
print(f"  batches/epoch: {len(dataloader)}")
print("  PASSED")

DataLoader test:
  sat         : torch.Size([2, 12, 4, 64, 64])
  weather     : torch.Size([2, 72, 7])
  gauge       : torch.Size([2, 72, 3])
  static_maps : torch.Size([2, 5, 64, 64])
  flood_mask  : torch.Size([2, 1, 64, 64])
  severity    : torch.Size([2, 4])
  risk        : torch.Size([2])
  damage_mask : torch.Size([2, 64, 64])
  batches/epoch: 32
  PASSED


In [16]:
# first real training run -- 3 epochs on synthetic data
# goal: verify loss goes down, no NaN, memory stable

torch.cuda.empty_cache()

model     = FloodCastNet(config).cuda()
criterion = MultiTaskLoss(config).cuda()

optimizer = AdamW(
    model.parameters(),
    lr=config.training.learning_rate,
    weight_decay=config.training.weight_decay
)
scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)
scaler    = torch.amp.GradScaler('cuda')

Path(config.checkpoint_dir).mkdir(parents=True, exist_ok=True)

print("Training FloodCastNet (synthetic data -- 3 epochs)")
print("=" * 52)

history = []
for epoch in range(1, 4):
    t0       = time.time()
    avg_loss, sub_losses = train_one_epoch(
        model, optimizer, scheduler, scaler,
        criterion, dataloader, config, epoch
    )
    elapsed  = time.time() - t0
    mem      = torch.cuda.memory_allocated(0) / 1e9

    history.append(avg_loss)

    print(f"\nEpoch {epoch} summary:")
    print(f"  avg loss  : {avg_loss:.4f}")
    print(f"  time      : {elapsed:.1f}s")
    print(f"  GPU mem   : {mem:.2f} GB")
    for k, v in sub_losses.items():
        print(f"  {k:<10}: {v:.4f}")

    if epoch % config.training.save_every_n_epochs == 0:
        save_checkpoint(model, optimizer, epoch, avg_loss, config)

print("\n" + "=" * 52)
print(f"Loss trend: {[f'{l:.4f}' for l in history]}")
if history[-1] < history[0]:
    print("Loss going DOWN -- training working correctly")
else:
    print("NOTE: synthetic data, loss direction may vary")
print("FloodCastNet training loop -- verified")

/tmp/ipykernel_24/1436218676.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, depth)


Training FloodCastNet (synthetic data -- 3 epochs)


/tmp/ipykernel_24/1046107372.py:92: UserWarning: Using a target size (torch.Size([2, 1, 1, 1])) that is different to the input size (torch.Size([2, 1, 64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  unc_loss = F.mse_loss(preds["aleatoric_unc"], actual_error)


  epoch 1 step 000 loss=2.8716 mem=1.1GB
  epoch 1 step 010 loss=2.8830 mem=1.1GB
  epoch 1 step 020 loss=2.8794 mem=1.2GB
  epoch 1 step 030 loss=2.8842 mem=1.2GB

Epoch 1 summary:
  avg loss  : 2.8850
  time      : 6.0s
  GPU mem   : 1.14 GB
  sp1       : 1.3402
  physics   : 0.0658
  sp2       : 0.0659
  sp3       : 0.8323
  sp4       : 0.5790
  sp8       : 0.0018
  epoch 2 step 000 loss=2.8666 mem=1.2GB
  epoch 2 step 010 loss=2.8584 mem=1.2GB
  epoch 2 step 020 loss=2.8637 mem=1.2GB
  epoch 2 step 030 loss=2.8605 mem=1.2GB

Epoch 2 summary:
  avg loss  : 2.8589
  time      : 5.0s
  GPU mem   : 1.14 GB
  sp1       : 1.3331
  physics   : 0.0535
  sp2       : 0.0667
  sp3       : 0.8300
  sp4       : 0.5740
  sp8       : 0.0016
  epoch 3 step 000 loss=2.8240 mem=1.2GB
  epoch 3 step 010 loss=2.8675 mem=1.2GB
  epoch 3 step 020 loss=2.8713 mem=1.2GB
  epoch 3 step 030 loss=2.8592 mem=1.2GB

Epoch 3 summary:
  avg loss  : 2.8601
  time      : 4.9s
  GPU mem   : 1.14 GB
  sp1       : 1.

In [17]:
# warning: actual_error shape (2,1,1,1) vs aleatoric_unc (2,1,64,64)
# fix: expand actual_error to match spatial dimensions

class MultiTaskLoss(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.weights    = cfg.training.loss_weights
        self.physics    = PhysicsLoss(weight=cfg.training.loss_weights["physics"])
        self.bce_flood  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]))
        self.ce_risk    = nn.CrossEntropyLoss()
        self.ce_damage  = nn.CrossEntropyLoss()
        self.mse_sev    = nn.MSELoss()

    def forward(self, preds, targets, static_maps):
        dem    = static_maps[:, 0:1]
        losses = {}

        if "flood_mask" in targets:
            losses["sp1"] = self.weights["sp1_flood_map"] * \
                self.bce_flood(preds["flood_map"], targets["flood_mask"])

        losses["physics"] = self.physics(preds["flood_map"], dem)

        if "severity" in targets:
            losses["sp2"] = self.weights["sp2_severity"] * \
                self.mse_sev(preds["severity"], targets["severity"])

        if "risk" in targets:
            losses["sp3"] = self.weights["sp3_risk"] * \
                self.ce_risk(preds["risk"], targets["risk"])

        if "damage_mask" in targets:
            losses["sp4"] = self.weights["sp4_damage"] * \
                self.ce_damage(preds["damage_map"], targets["damage_mask"])

        if "flood_mask" in targets:
            with torch.no_grad():
                # pixel-level error -- keep spatial dims (B,1,H,W)
                actual_error = (
                    torch.sigmoid(preds["flood_map"]) - targets["flood_mask"]
                ).abs()   # (B, 1, H, W) -- no mean, keep spatial

            # now both are (B, 1, H, W) -- no broadcasting issue
            unc_loss = F.mse_loss(preds["aleatoric_unc"], actual_error)
            losses["sp8"] = self.weights["sp8_uncertainty"] * unc_loss

        total = sum(losses.values())
        return total, losses


# quick verify -- no warning should appear
criterion = MultiTaskLoss(config).cuda()
B, H, W   = 2, 64, 64
static_in  = torch.randn(B, 5, H, W).cuda()
targets    = {
    "flood_mask"  : (torch.rand(B,1,H,W) < 0.1).float().cuda(),
    "severity"    : torch.rand(B,4).cuda(),
    "risk"        : torch.randint(0,4,(B,)).cuda(),
    "damage_mask" : torch.randint(0,3,(B,H,W)).cuda()
}

with torch.amp.autocast('cuda'):
    out = model(**{
        "sat"         : torch.randn(B,12,4,H,W).cuda(),
        "weather"     : torch.randn(B,72,7).cuda(),
        "gauge"       : torch.randn(B,72,3).cuda(),
        "static_maps" : static_in
    })
    loss, ld = criterion(out, targets, static_in)

print("MultiTaskLoss (fixed):")
print(f"  total : {loss.item():.4f}")
for k,v in ld.items():
    print(f"  {k:<10}: {v.item():.4f}")
assert not torch.isnan(loss)
print("  No shape warnings -- FIXED")

MultiTaskLoss (fixed):
  total : 2.7828
  sp1       : 1.3277
  physics   : 0.0367
  sp2       : 0.0633
  sp3       : 0.7840
  sp4       : 0.5687
  sp8       : 0.0022
  No shape warnings -- FIXED


In [18]:
# save full model code as a .py file
# this goes into your GitHub repo

import os
from pathlib import Path

save_dir = Path("/kaggle/working/FloodCastNet/models")
save_dir.mkdir(parents=True, exist_ok=True)

# save checkpoint
ckpt_path = f"{config.checkpoint_dir}/floodcastnet_v1.pt"
Path(config.checkpoint_dir).mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state"  : model.state_dict(),
    "config"       : asdict(config),
    "total_params" : sum(p.numel() for p in model.parameters()),
    "version"      : "1.0.0",
    "architecture" : "FloodCastNet-9SP",
}, ckpt_path)

size_mb = os.path.getsize(ckpt_path) / 1e6
print(f"Model saved:")
print(f"  path    : {ckpt_path}")
print(f"  size    : {size_mb:.1f} MB")
print(f"  params  : {sum(p.numel() for p in model.parameters()):,}")

# verify reload works
ckpt   = torch.load(ckpt_path, map_location='cuda')
model2 = FloodCastNet(config).cuda()
model2.load_state_dict(ckpt["model_state"])
model2.eval()

with torch.no_grad(), torch.amp.autocast('cuda'):
    out2 = model2(**{
        "sat"         : torch.randn(2,12,4,64,64).cuda(),
        "weather"     : torch.randn(2,72,7).cuda(),
        "gauge"       : torch.randn(2,72,3).cuda(),
        "static_maps" : torch.randn(2,5,64,64).cuda()
    })

print(f"  reload  : OK -- {len(out2)} outputs verified")
print(f"  ready for real data training")

Model saved:
  path    : /kaggle/working/FloodCastNet/checkpoints/floodcastnet_v1.pt
  size    : 18.8 MB
  params  : 4,660,313
  reload  : OK -- 14 outputs verified
  ready for real data training


/tmp/ipykernel_24/1436218676.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, depth)
